## SRP526713 / GSE275522

**paper:** [PMID: 39874536](https://pmc.ncbi.nlm.nih.gov/articles/PMC11823453/) - Contaminant-Associated Disruption of the Skin Transcriptome in the Endangered St. Lawrence Estuary Beluga, 2025

**date, curator:** 2026-04-14, Sara Carsanaro

**notes**

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Skin,UBERON:0000014,zone of skin,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP526713"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 69/69 [01:16<00:00,  1.10s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-21-15,,,,,,TRANSCRIPTOMIC,PCR
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-20-02,,,,,,TRANSCRIPTOMIC,PCR
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-09,,,,,,TRANSCRIPTOMIC,PCR
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-08,,,,,,TRANSCRIPTOMIC,PCR
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1683,,,,,,TRANSCRIPTOMIC,PCR
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1681,,,,,,TRANSCRIPTOMIC,PCR
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1679,,,,,,TRANSCRIPTOMIC,PCR
7,SRX25720286,SRP526713,Illumina NovaSeq 6000,SRS22361119,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1677,SAMN43107270,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1677,,,,,,TRANSCRIPTOMIC,PCR
8,SRX25720285,SRP526713,Illumina NovaSeq 6000,SRS22361118,,,,,,Skin,Adult,,,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Skin']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000014'
library.loc[:,'anatName'] = 'zone of skin'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-21-15,,,,,,TRANSCRIPTOMIC,PCR
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-20-02,,,,,,TRANSCRIPTOMIC,PCR
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-09,,,,,,TRANSCRIPTOMIC,PCR
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-08,,,,,,TRANSCRIPTOMIC,PCR
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1683,,,,,,TRANSCRIPTOMIC,PCR
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1681,,,,,,TRANSCRIPTOMIC,PCR
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1679,,,,,,TRANSCRIPTOMIC,PCR
7,SRX25720286,SRP526713,Illumina NovaSeq 6000,SRS22361119,UBERON:0000014,zone of skin,,,,Skin,Adult,perfect match,not documented,,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,rib

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-21-15,,,,,,TRANSCRIPTOMIC,PCR
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-20-02,,,,,,TRANSCRIPTOMIC,PCR
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-09,,,,,,TRANSCRIPTOMIC,PCR
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-08,,,,,,TRANSCRIPTOMIC,PCR
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1683,,,,,,TRANSCRIPTOMIC,PCR
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1681,,,,,,TRANSCRIPTOMIC,PCR
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted 

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'KAPA RNA HyperPrep Kit with RiboErase'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'ribo-minus'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-21-15,,,,,,TRANSCRIPTOMIC,PCR
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-20-02,,,,,,TRANSCRIPTOMIC,PCR
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-09,,,,,,TRANSCRIPTOMIC,PCR
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,,Eastern Beaufort Sea,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-08,,,,,,TRANSCRIPTOMIC,PCR
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1683,,,,,,TRANSCRIPTOMIC,PCR
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1681,,,,,,TRANSCRIPTOMIC,PCR
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,,St. Lawrence Estuary,,,14/04/2026,"Total RNA were extracted 

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-14'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,,Eastern Beaufort Sea,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-21-15,,,,,,TRANSCRIPTOMIC,PCR
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,,Eastern Beaufort Sea,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-20-02,,,,,,TRANSCRIPTOMIC,PCR
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,,Eastern Beaufort Sea,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-09,,,,,,TRANSCRIPTOMIC,PCR
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,,Eastern Beaufort Sea,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,ARHI-18-08,,,,,,TRANSCRIPTOMIC,PCR
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,,St. Lawrence Estuary,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1683,,,,,,TRANSCRIPTOMIC,PCR
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,,St. Lawrence Estuary,,SAC,2026-04-14,"Total RNA were extracted from TRIzol then reverse-transcripted into cDNA, the library used was rRNA-depleted stranded (HMR) and the adapter used was NEBNext_dual",,DLB-1681,,,,,,TRANSCRIPTOMIC,PCR
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,,St. Lawrence Estuary,,SAC,2026-04-14,"Tota

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 39874536'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
1,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
2,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
3,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
4,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14
5,SRX25720288,SRP526713,Illumina NovaSeq 6000,SRS22361122,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1681,SAMN43107272,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14
6,SRX25720287,SRP526713,Illumina NovaSeq 6000,SRS22361120,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1679,SAMN43107271,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14
7,SRX25720286,SRP526713,Illumina NovaSeq 6000,SRS22361119,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1677,SAMN43107270,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14
8,SRX25720285,SRP526713,Illumina NovaSeq 6000,SRS22361118,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1675,SAMN43107269,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14
9,SRX25720284,SRP526713,Illumina NovaSeq 6000,SRS22361117,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1672,SAMN43107268,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP526713,"Delphinapterus leucas skin transcriptome sequencing and assembly, and comparison between two Canadian populations","The St. Lawrence Estuary (SLE) beluga (Delphinapterus leucas) population in Canada is endangered, and endocrine disrupting contaminants such as polychlorinated biphenyls (PCBs), polybrominated diphenyl ethers (PBDEs), and other halogenated flame retardants (HFRs) have been identified as a threat to the recovery of this population. While targeted approaches such as quantitative reverse transcription PCR (RT-qPCR) have been widely used to assess the impacts of contaminants on marine mammals, including SLE beluga, few studies have employed transcriptomics. Here, we (1) evaluate the skin transcriptome profiles of adult male SLE beluga and adult males from a Arctic population less exposed to contaminants (Eastern Beaufort Sea; EBS) used as a reference to identify gene transcripts and biological pathways associated with blubber concentrations of organic contaminants (i.e., PCBs, PBDEs and other HFRs), and (2) assess correlations between several gene transcripts previously identified as biomarkers of organic contaminants in marine mammals and organohalogen concentrations in both populations and estimate threshold values in beluga skin for potential biological effects. Results will provide new and valuable knowledge that identify biological pathways associated with organic contaminant exposure in beluga, which may serve as predictors for higher-level health effects in at-risk populations such as SLE beluga.",SRA,,,,KAPA RNA HyperPrep Kit with RiboErase,full_length,,PRJNA1146570,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

69

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP526713,"Delphinapterus leucas skin transcriptome sequencing and assembly, and comparison between two Canadian populations","The St. Lawrence Estuary (SLE) beluga (Delphinapterus leucas) population in Canada is endangered, and endocrine disrupting contaminants such as polychlorinated biphenyls (PCBs), polybrominated diphenyl ethers (PBDEs), and other halogenated flame retardants (HFRs) have been identified as a threat to the recovery of this population. While targeted approaches such as quantitative reverse transcription PCR (RT-qPCR) have been widely used to assess the impacts of contaminants on marine mammals, including SLE beluga, few studies have employed transcriptomics. Here, we (1) evaluate the skin transcriptome profiles of adult male SLE beluga and adult males from a Arctic population less exposed to contaminants (Eastern Beaufort Sea; EBS) used as a reference to identify gene transcripts and biological pathways associated with blubber concentrations of organic contaminants (i.e., PCBs, PBDEs and other HFRs), and (2) assess correlations between several gene transcripts previously identified as biomarkers of organic contaminants in marine mammals and organohalogen concentrations in both populations and estimate threshold values in beluga skin for potential biological effects. Results will provide new and valuable knowledge that identify biological pathways associated with organic contaminant exposure in beluga, which may serve as predictors for higher-level health effects in at-risk populations such as SLE beluga.",SRA,total,Bgee 1K,69,KAPA RNA HyperPrep Kit with RiboErase,full_length,,PRJNA1146570,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
experiment.loc[:,'GSE'] = 'GSE275522'
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39874536'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11823453/'
experiment.loc[:,'DOI'] = '10.1021/acs.est.4c08272'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP526713,"Delphinapterus leucas skin transcriptome sequencing and assembly, and comparison between two Canadian populations","The St. Lawrence Estuary (SLE) beluga (Delphinapterus leucas) population in Canada is endangered, and endocrine disrupting contaminants such as polychlorinated biphenyls (PCBs), polybrominated diphenyl ethers (PBDEs), and other halogenated flame retardants (HFRs) have been identified as a threat to the recovery of this population. While targeted approaches such as quantitative reverse transcription PCR (RT-qPCR) have been widely used to assess the impacts of contaminants on marine mammals, including SLE beluga, few studies have employed transcriptomics. Here, we (1) evaluate the skin transcriptome profiles of adult male SLE beluga and adult males from a Arctic population less exposed to contaminants (Eastern Beaufort Sea; EBS) used as a reference to identify gene transcripts and biological pathways associated with blubber concentrations of organic contaminants (i.e., PCBs, PBDEs and other HFRs), and (2) assess correlations between several gene transcripts previously identified as biomarkers of organic contaminants in marine mammals and organohalogen concentrations in both populations and estimate threshold values in beluga skin for potential biological effects. Results will provide new and valuable knowledge that identify biological pathways associated with organic contaminant exposure in beluga, which may serve as predictors for higher-level health effects in at-risk populations such as SLE beluga.",SRA,total,Bgee 1K,69,KAPA RNA HyperPrep Kit with RiboErase,full_length,GSE275522,PRJNA1146570,39874536,https://pmc.ncbi.nlm.nih.gov/articles/PMC11823453/,10.1021/acs.est.4c08272,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [22]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
61752,SRX3291905,SRP120137,NextSeq 500,SRS2601092,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,F,,,9749,TruSeq Stranded mRNA,full_length,polyA,,2,DLBB-08-02,SAMN07782284,,,,,,,2 samples from https://www.frontiersin.org/jou...,,,SAC,2026-04-14
61753,SRX3291904,SRP120137,NextSeq 500,SRS2601092,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,F,,,9749,TruSeq Stranded mRNA,full_length,polyA,,2,DLBB-08-02,SAMN07782284,,,,,,,2 samples from https://www.frontiersin.org/jou...,,,SAC,2026-04-14
61754,SRX25720246,SRP526713,Illumina NovaSeq 6000,SRS22361079,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-21-15,SAMN43107234,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
61755,SRX25720292,SRP526713,Illumina NovaSeq 6000,SRS22361124,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-20-02,SAMN43107214,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
61756,SRX25720291,SRP526713,Illumina NovaSeq 6000,SRS22361125,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-09,SAMN43107213,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
61757,SRX25720290,SRP526713,Illumina NovaSeq 6000,SRS22361123,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,ARHI-18-08,SAMN43107212,,,,,,,PMID: 39874536,Eastern Beaufort Sea,,SAC,2026-04-14
61758,SRX25720289,SRP526713,Illumina NovaSeq 6000,SRS22361121,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,M,,,9749,KAPA RNA HyperPrep Kit with RiboErase,full_length,ribo-minus,,,DLB-1683,SAMN43107273,,,,,,,PMID: 39874536,St. Lawrence Estuary,,SAC,2026-04-14


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1181,SRP476418,Transcriptomic reads of Adineta vaga under rad...,Raw transcriptomic reads of Adineta vaga indiv...,SRA,total,Bgee 1K,61,TruSeq Stranded mRNA,full_length,,PRJNA962496,38273318,https://pmc.ncbi.nlm.nih.gov/articles/PMC10809...,10.1186/s12915-023-01807-8,,
1182,SRP120137,Delphinapterus leucas skin transcriptome seque...,Skin samples were collected from the beluga po...,SRA,total,Bgee 1K,8,TruSeq Stranded mRNA,full_length,,PRJNA414234,,https://www.frontiersin.org/journals/marine-sc...,10.3389/fmars.2024.1282210,,"no PMID for this paper, also for some reason t..."
1183,SRP526713,Delphinapterus leucas skin transcriptome seque...,The St. Lawrence Estuary (SLE) beluga (Delphin...,SRA,total,Bgee 1K,69,KAPA RNA HyperPrep Kit with RiboErase,full_length,GSE275522,PRJNA1146570,39874536,https://pmc.ncbi.nlm.nih.gov/articles/PMC11823...,10.1021/acs.est.4c08272,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../1_reject_experiment/out/
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 5e1a052] adding annotated bulk experiment SRP526713
 2 files changed, 70 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.77 KiB | 965.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   3fd8aa0..5e1a052  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push